In [17]:
import pandas as pd

In [18]:
# --- CONFIGURACIÓN ---
# archivo CSV original con todos los datos de contaminantes.
archivo_original_contaminantes = 'datos_consolidados_ordenados.csv'

# archivo que hice en la Fase 3 con las coordenadas.
archivo_coordenadas = 'antenas_con_coordenadas.csv'

# archivo final que se creará con todo unido.
archivo_salida_final = 'datos_consolidados_ordenados_concoords.csv'
# ---------------------

In [22]:
try:
    # 1: Cargar ambos archivos CSV en DataFrames de Pandas.
    df_contaminantes = pd.read_csv(archivo_original_contaminantes)
    df_coordenadas = pd.read_csv(archivo_coordenadas)

    print("Archivos cargados correctamente.")
    print(f"-> Archivo de contaminantes tiene {len(df_contaminantes)} filas.")
    print(f"-> Archivo de coordenadas tiene {len(df_coordenadas)} filas.")

    # 2: Creamos una "llave de unión" limpia en el DataFrame de contaminantes.
    # Limpiamos la columna 'fuente_archivo' para que coincida con el formato de la columna 'antena'.
    # NOTA importante para no olvidar: Esta limpieza debe ser idéntica a la que hice en la Fase 2.
    df_contaminantes['antena_limpia'] = (
        df_contaminantes['fuente_archivo']
        .str.replace('-air-quality.csv', '', regex=False)
        .str.replace(',', '', regex=False)
        .str.replace('-', ' ', regex=False)
        .str.strip()
    )
    
    print("\nSe ha creado una columna temporal 'antena_limpia' para la unión.")

    # 3: hacer el "merge" (la unión) de los dos DataFrames.
    # Usamos un 'left merge' para asegurarnos de conservar todas las filas del archivo original de contaminantes.
    df_final = pd.merge(
        df_contaminantes, 
        df_coordenadas, 
        left_on='antena_limpia',  # Columna clave en el archivo de la izquierda
        right_on='antena',       # Columna clave en el archivo de la derecha
        how='left'               # Tipo de unión
    )

    print("La unión (merge) se ha completado.")


    

    # 4: Limpiar el DataFrame final.
    # Eliminamos las columnas que usamos para la unión y que ya no necesitamos.
    df_final = df_final.drop(columns=['antena_limpia'])
    
    print("Se han eliminado las columnas auxiliar 'antena_limpia'.")

    # 5: Guardar el resultado en un nuevo archivo CSV.
    df_final.to_csv(archivo_salida_final, index=False)

    print(f"\n excelente se ha creado el archivo '{archivo_salida_final}'.")
    print(f"El nuevo archivo tiene {len(df_final)} filas y ahora incluye las columnas 'antena' 'latitud' y 'longitud'.")

except FileNotFoundError as e:
    print(f"Error: No se pudo encontrar un archivo. Revisa el nombre: {e.filename}")
except KeyError as e:
    print(f"Error: No se encontró una columna necesaria. Revisa que el nombre de la columna sea correcto: {e}")


Archivos cargados correctamente.
-> Archivo de contaminantes tiene 217854 filas.
-> Archivo de coordenadas tiene 86 filas.

Se ha creado una columna temporal 'antena_limpia' para la unión.
La unión (merge) se ha completado.
Se han eliminado las columnas auxiliar 'antena_limpia'.

 excelente se ha creado el archivo 'datos_consolidados_ordenados_concoords.csv'.
El nuevo archivo tiene 217854 filas y ahora incluye las columnas 'antena' 'latitud' y 'longitud'.


# Diagnóstico para identificar columnas con caracteres diferentes

#### esta parte del codigo es un diagnostico por que hubo problemas para cargas las coordenadas de centro chih1 chihuahua estatal y sur chih1 chihuahua estatal. después de encontrar las diferencias se corrigó manualmente y regresamos al script anterior para volver a ejecutarlo y se solucionó el problema.

problema identificado: faltaba un espacio

In [16]:
import pandas as pd

# --- CÓDIGO DE DIAGNÓSTICO ---

# Cargamos los dos archivos clave
df_coordenadas = pd.read_csv('antenas_con_coordenadas.csv')
df_contaminantes = pd.read_csv('datos_consolidados_ordenados.csv')

# 1. Limpiamos la columna del archivo de contaminantes, igual que en el script de la Fase 4 y 2
df_contaminantes['antena_limpia'] = (
    df_contaminantes['fuente_archivo']
    .str.replace('-air-quality.csv', '', regex=False)
    .str.replace(',', '', regex=False)
    .str.replace('-', ' ', regex=False) #aqui puede estar el problema con 2 antenas por eso se pone doble espacio
    .str.strip()
)

# 2. Buscamos las versiones de los nombres en ambos archivos

# Versión del nombre en el archivo de coordenadas (la que SÍ tiene lat/lon)
nombre_en_coordenadas_centro = df_coordenadas[df_coordenadas['antena'].str.contains("centro chih1", na=False)]['antena'].iloc[0]
nombre_en_coordenadas_sur = df_coordenadas[df_coordenadas['antena'].str.contains("sur chih1", na=False)]['antena'].iloc[0]

# Versión del nombre en el archivo de contaminantes (la que NO encontró coincidencia)
nombre_limpio_contaminantes_centro = df_contaminantes[df_contaminantes['antena_limpia'].str.contains("centro chih1", na=False)]['antena_limpia'].iloc[0]
nombre_limpio_contaminantes_sur = df_contaminantes[df_contaminantes['antena_limpia'].str.contains("sur chih1", na=False)]['antena_limpia'].iloc[0]


# 3. Imprimimos los nombres rodeados de [ ] para poder ver los espacios ocultos
print("--- Comparando 'centro chih1 chihuahua estatal' ---")
print(f"Versión en 'antenas_con_coordenadas.csv': [{nombre_en_coordenadas_centro}]")
print(f"Versión limpia en archivo de contaminantes:  [{nombre_limpio_contaminantes_centro}]")
print(f"¿Son idénticos?: {nombre_en_coordenadas_centro == nombre_limpio_contaminantes_centro}")

print("\n--- Comparando 'sur chih1 chihuahua estatal' ---")
print(f"Versión en 'antenas_con_coordenadas.csv': [{nombre_en_coordenadas_sur}]")
print(f"Versión limpia en archivo de contaminantes:  [{nombre_limpio_contaminantes_sur}]")
print(f"¿Son idénticos?: {nombre_en_coordenadas_sur == nombre_limpio_contaminantes_sur}")

--- Comparando 'centro chih1 chihuahua estatal' ---
Versión en 'antenas_con_coordenadas.csv': [centro chih1 chihuahua  estatal]
Versión limpia en archivo de contaminantes:  [centro chih1 chihuahua  estatal]
¿Son idénticos?: True

--- Comparando 'sur chih1 chihuahua estatal' ---
Versión en 'antenas_con_coordenadas.csv': [sur chih1 chihuahua  estatal]
Versión limpia en archivo de contaminantes:  [sur chih1 chihuahua  estatal]
¿Son idénticos?: True
